In [6]:
import json
import pandas as pd
from pathlib import Path


def build_seq_id_dict(s, prefix, save_path):
    # 获取唯一项
    uniq = sorted(s.drop_duplicates())
    d = {seq: f"{prefix}_{i}" for i, seq in enumerate(uniq, start=1)}

    save_path = Path(save_path)
    with save_path.open("w", encoding="utf-8") as f:
        json.dump(d, f, ensure_ascii=False, indent=2)

    return d

def make_seasonal_split(All_data, sheet_number, group_columns, agg_dict, save_path):
    s1 = All_data['sheet'].astype(str).str.split('-', n=1, expand=True)[0].astype(int)
    train_data_seasonal = All_data[s1 < sheet_number].groupby(group_columns).agg(agg_dict).reset_index()
    test_data_seasonal  = All_data[s1 == sheet_number].groupby(group_columns).agg(agg_dict).reset_index()

    print(train_data_seasonal.shape)
    print(test_data_seasonal.shape)

    train_data_seasonal.to_csv(save_path + '/train.csv', index=False)
    test_data_seasonal.to_csv(save_path + '/test.csv', index=False)

    return train_data_seasonal, test_data_seasonal

def make_artificial_data(train_data):
    serum = train_data[['seq_id_a', 'seq_id_b', 'seq_a', 'seq_b', 'serumPassCat']]
    virus = train_data[['seq_id_c', 'seq_id_d', 'seq_c', 'seq_d', 'virusPassCat']]
    virus.columns = ['seq_id_a', 'seq_id_b', 'seq_a', 'seq_b', 'serumPassCat']

    artificial_data = pd.concat([serum, virus], axis=1).drop_duplicates()
    artificial_data.columns = ['seq_id_a', 'seq_id_b', 'seq_a', 'seq_b', 'serumPassCat',
                                'seq_id_c', 'seq_id_d', 'seq_c', 'seq_d', 'virusPassCat']
    artificial_data['label'] = 0
    return artificial_data

def dict_to_fasta(dict, out_fasta) -> None:
    with open(out_fasta, "w", encoding="utf-8") as f:
        for key, value in dict.items():
            f.write(f">{value}\n{key}\n")
            
def fasta_to_dict(fasta_path: str):
    d = {}
    header = None

    with open(fasta_path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            if line.startswith(">"):
                header = line[1:].split()[0]   # 取第一个token做key
                d[header] = ""
            else:
                d[header] += line              # 拼接多行序列
    return d

In [2]:
H1N1_origin = pd.read_csv('./raw/data4model(H1N1).csv')
H3N2_origin = pd.read_csv('./raw/data4model(H3N2).csv')
selected_columns = ['seq_a', 'seq_b', 'seq_c', 'seq_d', 'serumPassCat', 'virusPassCat', 'serumName',
                    'virusName', 'label', 'serumDate', 'serumType', 'virusDate', 'serumIslID', 'virusIslID', 'sheet']
modified_columns = ['seq_a', 'seq_b', 'seq_c', 'seq_d', 'serumPassCat', 'virusPassCat', 'serumName',
                    'virusName', 'label', 'serumDate', 'Type', 'virusDate', 'serumIslID', 'virusIslID', 'sheet']
                    
H1N1_data = H1N1_origin[selected_columns]
H1N1_data.columns = modified_columns
H3N2_data = H3N2_origin[selected_columns]
H3N2_data.columns = modified_columns

All_data = pd.concat([H1N1_data, H3N2_data]).reset_index(drop=True)

In [3]:
HA_seqs = pd.concat([All_data["seq_a"], All_data["seq_c"]], ignore_index=True)
NA_seqs = pd.concat([All_data["seq_b"], All_data["seq_d"]], ignore_index=True)

HA_mapping = build_seq_id_dict(HA_seqs, "HA", "./HA_map.json")
NA_mapping = build_seq_id_dict(NA_seqs, "NA", "./NA_map.json")

All_data["seq_id_a"] = All_data["seq_a"].map(HA_mapping)
All_data["seq_id_c"] = All_data["seq_c"].map(HA_mapping)
All_data["seq_id_b"] = All_data["seq_b"].map(NA_mapping)
All_data["seq_id_d"] = All_data["seq_d"].map(NA_mapping)
All_data = All_data[['seq_id_a', 'seq_id_b', 'seq_id_c', 'seq_id_d'] + modified_columns]

In [ ]:
# dict_to_fasta(HA_mapping, "HA_orig.fasta")
# dict_to_fasta(NA_mapping, "NA_orig.fasta")

In [7]:
fasta_path = "./HA_aligned.fasta"
aligned_HA = fasta_to_dict(fasta_path)

In [8]:
All_data["serumHA"] = All_data["seq_id_a"].map(aligned_HA)
All_data["virusHA"] = All_data["seq_id_c"].map(aligned_HA)

In [9]:
All_data.to_csv('./processed/All.csv', index=False)

In [2]:
All_data = pd.read_csv('./processed/All.csv')

In [11]:
group_columns = ['seq_a', 'seq_b', 'seq_c', 'seq_d', 'serumPassCat', 'virusPassCat']
agg_dict = {c: 'first' for c in All_data.columns if c not in group_columns}
agg_dict['label'] = 'mean'

In [12]:
train_2023NH, test_2023NH = make_seasonal_split(All_data, 39, group_columns, agg_dict, './processed/test_2023NH')
train_2023SH, test_2023SH = make_seasonal_split(All_data, 40, group_columns, agg_dict, './processed/test_2023SH')
train_2024NH, test_2024NH = make_seasonal_split(All_data, 41, group_columns, agg_dict, './processed/test_2024NH')
train_2024SH, test_2024SH = make_seasonal_split(All_data, 42, group_columns, agg_dict, './processed/test_2024SH')
train_2025NH, test_2025NH = make_seasonal_split(All_data, 43, group_columns, agg_dict, './processed/test_2025NH')
train_2025SH, test_2025SH = make_seasonal_split(All_data, 44, group_columns, agg_dict, './processed/test_2025SH')

(62632, 21)
(3502, 21)
(65847, 21)
(6626, 21)
(71915, 21)
(2774, 21)
(74598, 21)
(7694, 21)
(81952, 21)
(4071, 21)
(85780, 21)
(4592, 21)


In [10]:
make_artificial_data(train_2024NH).to_csv('./processed/test_2024NH/artificial_data.csv', index=False)
make_artificial_data(train_2024SH).to_csv('./processed/test_2024SH/artificial_data.csv', index=False)
make_artificial_data(train_2025NH).to_csv('./processed/test_2025NH/artificial_data.csv', index=False)
make_artificial_data(train_2025SH).to_csv('./processed/test_2025SH/artificial_data.csv', index=False)